In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

filePath = 'combined.csv'
df = pd.read_csv(filePath)

drop_cols = ['Time', 'Attendance', 'HHW', 'AHW', 'HO', 'AO', 'Div']
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

def date_format_type(date_str):
    if not isinstance(date_str, str):
        return "not_a_string"
    patterns = {
        "%d/%m/%y": r"^\d{2}/\d{2}/\d{2}$",
        "%d/%m/%Y": r"^\d{2}/\d{2}/\d{4}$",
        "%Y-%m-%d": r"^\d{4}-\d{2}-\d{2}$",
        "%m-%d-%Y": r"^\d{2}-\d{2}-\d{4}$",
        "%Y/%m/%d": r"^\d{4}/\d{2}/\d{2}$",
    }
    for fmt, pat in patterns.items():
        if re.match(pat, date_str):
            return fmt
    return "unknown"

df['DateFormat'] = df['Date'].apply(date_format_type)
def parse_dates(row):
    date_str = row['Date']
    if isinstance(date_str, str):
        try:
            return pd.to_datetime(date_str, format='%d/%m/%y')
        except ValueError:
            try:
                return pd.to_datetime(date_str, format='%d/%m/%Y')
            except ValueError:
                return pd.NaT
    else:
        return pd.NaT

df['Date'] = df.apply(parse_dates, axis=1)
df = df.drop(columns=['DateFormat'], errors='ignore')
df = df[df['Date'] >= pd.Timestamp('2000-08-18')]
df = df.reset_index(drop=True)

def get_season(date):
    if pd.isnull(date):
        return np.nan
    year = date.year
    month = date.month
    if month >= 8: 
        return f"{year}-{str(year+1)[-2:]}"
    else:
        return f"{year-1}-{str(year)[-2:]}"
df['Season'] = df['Date'].apply(get_season)
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek

odds_cols = [
    'IWH', 'IWD', 'IWA', 'WHH', 'WHD', 'WHA', 'B365H', 'B365D', 'B365A', 
    'B365>2.5', 'B365<2.5'
]
score_cols = [
    'FTHG', 'FTAG', 'HTHG', 'HTAG', 'HS', 'AS', 'HST', 'AST',  
    'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR'
]
for col in odds_cols + score_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.drop_duplicates()

core_odds = ['IWH','IWD','IWA','WHH','WHD','WHA','B365H','B365D','B365A']
df = df.dropna(subset=[c for c in core_odds if c in df.columns])
df = df.reset_index(drop=True)

df['TotalGoals'] = df['FTHG'] + df['FTAG']
df['GoalsOver2_5'] = (df['TotalGoals'] > 2.5).astype(int)
df['BTTS'] = ((df['FTHG'] > 0) & (df['FTAG'] > 0)).astype(int)
df['Home_2plus'] = (df['FTHG'] >= 2).astype(int)
df['Away_2plus'] = (df['FTAG'] >= 2).astype(int)

df = df.sort_values('Date')
df['HomeTeam_mean_FTHG'] = (
    df.groupby('HomeTeam')['FTHG'].transform(lambda x: x.shift(1).expanding().mean())
)
df['AwayTeam_mean_FTAG'] = (
    df.groupby('AwayTeam')['FTAG'].transform(lambda x: x.shift(1).expanding().mean())
)

df['HomeTeam_str'] = df['HomeTeam']
df['AwayTeam_str'] = df['AwayTeam']
df = pd.get_dummies(df, columns=['HomeTeam', 'AwayTeam'])
df = df.rename(columns={'HomeTeam_str': 'HomeTeam', 'AwayTeam_str': 'AwayTeam'})

def add_recent_form_features(df, n_matches=5):
    base = df.copy()
    base = base.sort_values('Date')
    home_df = base[['Date', 'HomeTeam', 'FTHG', 'FTAG']].rename(
        columns={'HomeTeam': 'Team', 'FTHG': 'GoalsFor', 'FTAG': 'GoalsAgainst'})
    away_df = base[['Date', 'AwayTeam', 'FTAG', 'FTHG']].rename(
        columns={'AwayTeam': 'Team', 'FTAG': 'GoalsFor', 'FTHG': 'GoalsAgainst'})
    results = pd.concat([home_df, away_df], ignore_index=True)
    results = results.sort_values(['Team', 'Date'])
   
    def get_points(row):
        return 3 if row['GoalsFor'] > row['GoalsAgainst'] else (1 if row['GoalsFor'] == row['GoalsAgainst'] else 0)
    results['Points'] = results.apply(get_points, axis=1)
    results['RollingGF'] = results.groupby('Team')['GoalsFor'].transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).mean())
    results['RollingGA'] = results.groupby('Team')['GoalsAgainst'].transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).mean())
    results['RollingPoints'] = results.groupby('Team')['Points'].transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).sum())
  
    def get_form(row, team_col):
        team = row[team_col]
        date = row['Date']
        row_form = results[(results['Team'] == team) & (results['Date'] < date)].sort_values('Date').tail(1)
        if row_form.empty:
            return pd.Series([np.nan, np.nan, np.nan])
        return row_form[['RollingGF', 'RollingGA', 'RollingPoints']].values[0]
    base[['HomeRecentGF', 'HomeRecentGA', 'HomeRecentPts']] = base.apply(
        lambda row: get_form(row, 'HomeTeam'), axis=1, result_type='expand')
    base[['AwayRecentGF', 'AwayRecentGA', 'AwayRecentPts']] = base.apply(
        lambda row: get_form(row, 'AwayTeam'), axis=1, result_type='expand')
    return base

df = add_recent_form_features(df, n_matches=5)
df = df.dropna(subset=['HomeRecentGF', 'AwayRecentGF'])

df = df.sort_values('Date')
h2h_home_wins = []
team_stats = {}
home_positions = []
away_positions = []
for idx, row in df.iterrows():
    home = row['HomeTeam']
    away = row['AwayTeam']
    match_date = row['Date']
    prev_matches = df[
        (((df['HomeTeam'] == home) & (df['AwayTeam'] == away)) |
         ((df['HomeTeam'] == away) & (df['AwayTeam'] == home)))
        & (df['Date'] < match_date)
    ].sort_values('Date', ascending=False).head(5)
    home_wins = ((prev_matches['HomeTeam'] == home) & (prev_matches['FTHG'] > prev_matches['FTAG'])).sum()
    h2h_home_wins.append(home_wins)
    # League table
    league_table = []
    for team, stats in team_stats.items():
        league_table.append({
            'team': team,
            'points': stats['points'],
            'gd': stats['gd'],
            'scored': stats['scored']
        })
    table_df = pd.DataFrame(league_table)
    if not table_df.empty:
        table_df = table_df.sort_values(['points', 'gd', 'scored'], ascending=[False, False, False])
        table_df['position'] = range(1, len(table_df) + 1)
        home_pos = table_df[table_df['team'] == home]['position'].values[0] if home in table_df['team'].values else len(table_df) + 1
        away_pos = table_df[table_df['team'] == away]['position'].values[0] if away in table_df['team'].values else len(table_df) + 1
    else:
        home_pos = away_pos = 1
    home_positions.append(home_pos)
    away_positions.append(away_pos)
    # Update stats
    home_goals = row['FTHG']
    away_goals = row['FTAG']
    for team in [home, away]:
        if team not in team_stats:
            team_stats[team] = {'points': 0, 'gd': 0, 'scored': 0}
    if home_goals > away_goals:
        team_stats[home]['points'] += 3
    elif home_goals < away_goals:
        team_stats[away]['points'] += 3
    else:
        team_stats[home]['points'] += 1
        team_stats[away]['points'] += 1
    team_stats[home]['gd'] += home_goals - away_goals
    team_stats[away]['gd'] += away_goals - home_goals
    team_stats[home]['scored'] += home_goals
    team_stats[away]['scored'] += away_goals

df['h2h_home_wins_last5'] = h2h_home_wins
df['home_league_position'] = home_positions
df['away_league_position'] = away_positions
df['position_diff'] = df['home_league_position'] - df['away_league_position']

N = 5
df['HomePts'] = np.where(df['FTHG'] > df['FTAG'], 3, np.where(df['FTHG'] == df['FTAG'], 1, 0))
df['AwayPts'] = np.where(df['FTAG'] > df['FTHG'], 3, np.where(df['FTAG'] == df['FTHG'], 1, 0))
df['HomeRecentPts'] = (
    df.groupby('HomeTeam')['HomePts'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['AwayRecentPts'] = (
    df.groupby('AwayTeam')['AwayPts'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['HomeGoalDiff'] = df['FTHG'] - df['FTAG']
df['AwayGoalDiff'] = df['FTAG'] - df['FTHG']
df['HomeRecentGoalDiff'] = (
    df.groupby('HomeTeam')['HomeGoalDiff'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['AwayRecentGoalDiff'] = (
    df.groupby('AwayTeam')['AwayGoalDiff'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['RecentGoalDiff'] = df['HomeRecentGoalDiff'] - df['AwayRecentGoalDiff']
df['HomeRecentShotsOnTarget'] = (
    df.groupby('HomeTeam')['HST'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['AwayRecentShotsOnTarget'] = (
    df.groupby('AwayTeam')['AST'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['RecentShotsOnTargetDiff'] = df['HomeRecentShotsOnTarget'] - df['AwayRecentShotsOnTarget']
df['PosDiff'] = df['home_league_position'] - df['away_league_position']

for col in ['B365>2.5', 'B365<2.5']:
    median = df[col].median()
    df[f'{col}_missing'] = df[col].isna().astype(int)
    df[col] = df[col].fillna(median)

df['B365>2.5_implied_prob'] = 1 / df['B365>2.5']
df['B365<2.5_implied_prob'] = 1 / df['B365<2.5']
df['OddsMargin'] = df['B365H'] / df['B365A']
df['OU_OddsMargin'] = df['B365>2.5'] / df['B365<2.5']
df['OverUnderRatio'] = df['B365>2.5'] / df['B365<2.5']

df['Weekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)
df['EarlySeason'] = (df['Month'] <= 3).astype(int)

def rolling_ref_aggression(subdf):
    agg = subdf[['HY', 'AY', 'HR', 'AR']].shift(1).sum(axis=1)
    return agg.rolling(10, min_periods=1).mean()
df = df.sort_values('Date')
df['RefereeAggression'] = (
    df.groupby('Referee').apply(lambda subdf: rolling_ref_aggression(subdf)).reset_index(level=0, drop=True)
)

df = df.replace([np.inf, -np.inf], np.nan)
for col in df.select_dtypes(include=['number']):
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include=['object', 'category']):
    df[col] = df[col].fillna(df[col].mode()[0])

df = df.drop_duplicates().reset_index(drop=True)

df.to_csv('final_football_model_data.csv', index=False)
print('Data pipeline complete. Saved as final_football_model_data.csv')

/var/folders/hh/pmnpfh4502s2v05fp58g729w0000gn/T/ipykernel_63249/4103366554.py:250: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('Referee').apply(lambda subdf: rolling_ref_aggression(subdf)).reset_index(level=0, drop=True)


Data pipeline complete. Saved as final_football_model_data.csv
